A SCORE MEANS NOTHING UNTIL WE KNOW WHAT AN EASY MODEL SCORES FOR THE DATSET. BASELINES PROVIDES US VALUES TO COMPARE AGAINST OUR MODEL.

In [6]:
import os
print(os.listdir("data/processed") if os.path.exists("data/processed") else "no folder yet")

['X_test.csv', 'X_train.csv', 'X_val.csv', 'y_test.csv', 'y_train.csv', 'y_val.csv']


In [7]:
#LOADING THE REQUIRED LIBRARIES AND SAVED SPLITS FROM PREPROCESSING 
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import joblib

X_train = pd.read_csv("data/processed/X_train.csv")
X_val   = pd.read_csv("data/processed/X_val.csv")
X_test  = pd.read_csv("data/processed/X_test.csv")
y_train = pd.read_csv("data/processed/y_train.csv").squeeze()
y_val   = pd.read_csv("data/processed/y_val.csv").squeeze()
y_test  = pd.read_csv("data/processed/y_test.csv").squeeze()

print(X_train.shape, X_val.shape, X_test.shape)

(170235, 30) (56745, 30) (56746, 30)


In [8]:
#BUILDING THE <DUMB> MODEL
#THIS MODEL WILL IGNORE THE DATSET AND WILL ALWAYS ANSWER NORMALLY. IT WILL BE HIGHLY ACCURATE AND ACTS AS AN ACCURACY TRAP THAT'S WHY WE USE OTHER METRICS.
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
pred = dummy.predict(X_val)

print("Accuracy:", accuracy_score(y_val, pred))
print("Frauds caught:", ((pred == 1) & (y_val == 1)).sum())
print("Total frauds in validation:", (y_val == 1).sum())

Accuracy: 0.9983434663847035
Frauds caught: 0
Total frauds in validation: 94


In [9]:
#DESIGNING THE SCORING FUNCTION: IT HELPS US TO MAKE SURE ALL THE MODELS ARE EVALUATED ON THE SAME METRICS AND WE CAN COMPARE THEM FAIRLY.
#PR-AUC: IS OUR MAIN METRIC. IT REWARDS THE MODEL THAT DETECTS THE FRAUDS NEAR THE TOP AND DOESN'T FLAG TOO MANY NORMAL TRANSACTIONS AS FRAUDS. IT'S A GOOD METRIC FOR IMBALANCED DATASETS.
#ROC-AUC: IS OUR SECONDARY METRIC. IT TEMDS TO LOOK HIGH ON IMBALANCED DATSETS.
from sklearn.metrics import average_precision_score, roc_auc_score

def report(name, y_true, scores):
    ap = average_precision_score(y_true, scores)
    roc = roc_auc_score(y_true, scores)
    print(f"{name:25s} PR-AUC = {ap:.4f}   ROC-AUC = {roc:.4f}")

In [ ]:
#A SUPERVISED BASELINE USING LOGISTIC REGRESSION